# Colab bootstrap — NNDL Saliency Project

Esegui queste celle in ordine ogni volta che apri una nuova sessione Colab.
**Regola d'oro:** salva SEMPRE i checkpoint su Drive, non solo sul disco della VM — la VM viene distrutta alla disconnessione.

## 1. Verifica GPU
Runtime > Change runtime type > GPU, poi esegui questa cella.

In [1]:
!nvidia-smi

Tue Sep 15 13:35:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Monta Google Drive (per checkpoint persistenti)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/nndl-saliency'
CHECKPOINT_DIR = f'{DRIVE_PROJECT_DIR}/checkpoints'

# Archivio persistente del dataset
DATA_ARCHIVE = f'{DRIVE_PROJECT_DIR}/salicon.tar'

# Dataset veloce usato dalla VM Colab
LOCAL_DATA_DIR = '/content/data_local'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print('Checkpoint dir:', CHECKPOINT_DIR)
print('Dataset archive:', DATA_ARCHIVE)
print('Local dataset:', LOCAL_DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint dir: /content/drive/MyDrive/nndl-saliency/checkpoints
Dataset archive: /content/drive/MyDrive/nndl-saliency/salicon.tar
Local dataset: /content/data_local


## 3. Clona/Agiorna il repository

In [3]:
import os

REPO_URL = 'https://github.com/markbtz/Project_NN.git'
REPO_DIR = '/content/nndl-saliency'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    %cd $REPO_DIR
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd $REPO_DIR

Cloning into '/content/nndl-saliency'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 30 (delta 5), reused 30 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 12.47 KiB | 4.16 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/nndl-saliency


## 4. Installa le dipendenze mancanti

In [4]:
!pip install -q -r requirements-colab.txt

## 5. Credenziali Kaggle (solo la prima volta / se non già su Drive)
Carica `kaggle.json` quando richiesto (Kaggle > Settings > Create New Token).

In [5]:
import os
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_drive = '/content/drive/MyDrive/nndl-saliency/kaggle.json'
if os.path.exists(kaggle_json_drive):
    !cp "$kaggle_json_drive" ~/.kaggle/kaggle.json
else:
    from google.colab import files
    uploaded = files.upload()  # carica kaggle.json
    !mv kaggle.json ~/.kaggle/kaggle.json
    !cp ~/.kaggle/kaggle.json "$kaggle_json_drive"  # salva su Drive per la prossima volta
!chmod 600 ~/.kaggle/kaggle.json

In [6]:
!kaggle datasets list -s salicon

ref                             title                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------  -------------------  ----------  --------------------------  -------------  ---------  ---------------  
hughiephan/salicon-mini         Salicon Mini           42143415  2024-09-18 08:55:53.797000             94          0  0.75             
harshgupta2411/salicon          SALICON              2726456977  2024-04-03 15:47:09.520000             97          0  0.0              
sarbojit3bhattachary/cognitive  Cognitive            5603129354  2025-07-19 18:45:05.943000              3          1  0.125            
roshan401/salicon               saliency in context  4264405363  2024-04-12 06:40:17.790000            607          2  0.5625           


## 6. Prepara SALICON su Drive (solo la prima volta)

Questa sezione scarica SALICON direttamente sul disco locale veloce di Colab, esegue l'audit, crea un unico `salicon.tar` e lo salva su Google Drive. Se `salicon.tar` esiste già, non riscarica nulla.

In [7]:
import os
import shutil
import subprocess

if os.path.exists(DATA_ARCHIVE):
    print('Archivio SALICON già presente su Drive:')
    print(DATA_ARCHIVE)
else:
    print('Archivio non presente: preparo SALICON per la prima volta.')

    # Rimuove solo un'eventuale copia locale incompleta della VM Colab.
    if os.path.exists(LOCAL_DATA_DIR):
        shutil.rmtree(LOCAL_DATA_DIR)
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

    print('\n1/4 - Download da Kaggle sul disco locale...')
    subprocess.run([
        'python', 'scripts/download_salicon.py',
        '--output_dir', LOCAL_DATA_DIR
    ], check=True)

    print('\n2/4 - Audit del dataset...')
    subprocess.run([
        'python', 'scripts/audit_dataset.py',
        '--data_dir', LOCAL_DATA_DIR
    ], check=True)

    LOCAL_ARCHIVE = '/content/salicon.tar'
    if os.path.exists(LOCAL_ARCHIVE):
        os.remove(LOCAL_ARCHIVE)

    print('\n3/4 - Creazione di salicon.tar...')
    subprocess.run([
        'tar', '-cf', LOCAL_ARCHIVE,
        '-C', LOCAL_DATA_DIR, '.'
    ], check=True)

    print('\n4/4 - Copia del singolo archivio su Google Drive...')
    shutil.copy2(LOCAL_ARCHIVE, DATA_ARCHIVE)
    os.remove(LOCAL_ARCHIVE)

    print('\nFatto. Archivio persistente salvato in:')
    print(DATA_ARCHIVE)

Archivio non presente: preparo SALICON per la prima volta.

1/4 - Download da Kaggle sul disco locale...

2/4 - Audit del dataset...

3/4 - Creazione di salicon.tar...

4/4 - Copia del singolo archivio su Google Drive...

Fatto. Archivio persistente salvato in:
/content/drive/MyDrive/nndl-saliency/salicon.tar


In [8]:
# Verifica che l'archivio persistente esista su Drive
if os.path.exists(DATA_ARCHIVE):
    size_gb = os.path.getsize(DATA_ARCHIVE) / (1024**3)
    print(f'OK: {DATA_ARCHIVE} ({size_gb:.2f} GB)')
else:
    print('ATTENZIONE: salicon.tar non è ancora presente su Drive.')

OK: /content/drive/MyDrive/nndl-saliency/salicon.tar (4.01 GB)


## 6bis. Prepara il dataset locale (a ogni nuova sessione)

Nelle sessioni successive NON riscaricare SALICON da Kaggle. Questa cella copia un solo archivio da Drive e lo estrae sul disco locale veloce della VM.

In [9]:
import os
import shutil
import subprocess
import time

LOCAL_ARCHIVE = '/content/salicon.tar'

if os.path.exists(LOCAL_DATA_DIR):
    print(f'{LOCAL_DATA_DIR} già presente. Salto preparazione.')
else:
    if not os.path.exists(DATA_ARCHIVE):
        raise FileNotFoundError(
            f'Archivio non trovato su Drive: {DATA_ARCHIVE}\n'
            'Esegui prima la sezione 6.'
        )

    t0 = time.time()

    print('Copio salicon.tar da Drive al disco locale...')
    shutil.copy2(DATA_ARCHIVE, LOCAL_ARCHIVE)

    print('Estraggo SALICON...')
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    subprocess.run([
        'tar', '-xf', LOCAL_ARCHIVE,
        '-C', LOCAL_DATA_DIR
    ], check=True)

    os.remove(LOCAL_ARCHIVE)
    print(f'Dataset pronto in {LOCAL_DATA_DIR} in {(time.time() - t0) / 60:.1f} minuti.')


/content/data_local già presente. Salto preparazione.


In [10]:
# Controllo finale della copia locale
!python scripts/audit_dataset.py --data_dir "$LOCAL_DATA_DIR"

Scansione di /content/data_local ...

CONTEGGIO FILE PER CATEGORIA (euristico, verificare a occhio)
  images                   : 20000
  density_maps             : 15000
  fixation_files           : 20000
  other_structured_files   : 0
  other_files              : 0

DOMANDA CRITICA: fixation coordinates disponibili?
  TROVATI 20000 file che sembrano fixation data.
  Esempi:
    /content/data_local/fixations/train/COCO_train2014_000000522862.mat
    /content/data_local/fixations/train/COCO_train2014_000000181909.mat
    /content/data_local/fixations/train/COCO_train2014_000000207880.mat
    /content/data_local/fixations/train/COCO_train2014_000000391480.mat
    /content/data_local/fixations/train/COCO_train2014_000000477226.mat
  -> Apri manualmente un file per confermare il formato (es. .mat con array Nx2).
  -> Se confermato: potete usare NSS e sAUC oltre a CC/SIM/KLD.

CORRISPONDENZA IMMAGINE <-> DENSITY MAP
  Coppie corrispondenti: 15000
  Immagini senza mappa:  5000
  Mappe senza 

### Routine per le sessioni future

Quando Colab assegna una nuova VM, esegui nell'ordine: **1 → 2 → 3 → 4 → 5 → 6bis**.  
La **sezione 6** serve normalmente una sola volta, finché `salicon.tar` rimane su Drive.


## 7. Da qui in poi: training / evaluation
Verranno aggiunte celle per `scripts/train.py` e `scripts/evaluate.py` a partire dal giorno 3.